## CUDA Basics
Compute Unified Device Architecture (CUDA) is a software programming model to allow users to use Nvidia GPUs for general purpose computing.

### CUDA Terminologies:

Host --> CPU

Device --> GPU


**Kernel in OS**

Bridge between software applications and computer hardware. It manages system resources, such as the CPU, memory, and devices, ensuring everything works together smoothly and efficiently. It handles tasks like running programs, accessing files, and connecting to devices like printers and keyboards.


**Kernel here**

Function to be executed on a GPU.
\__global__ means execute it on gpu
Kernel definition, function starts with \__global__ keyword (kernels called from cpu function like main(), are prefixed with \__global__, kernels called from within another kernel are prefixed with \__device__)



**Stream Multiprocessors (SM) and Stream Processors (SP)**

each gpu's computing unit is SM and each SM has multiple SPs where actual computation happens. SP is also called a CUDA core.



**Thread**

Single instance of execution, on one SP, one or more threads can be executed.



**Block**

Group of threads is called a block. On one SM, one or more blocks can be executed.



**Grid**

Group of blocks is called a grid. One grid is generated for one kernel on one GPU. Only one kernel can be executed at one time instance.



**Warp**

Number of threads in a block running simultaneously on a SM is called a warp.






**Malloc**

Purpose: Allocates a block of memory of a specified size in bytes from the heap.
In process heap

Initialization: The allocated memory is not initialized, and its contents are garbage values.

Syntax: void* malloc(size_t size);

int *ptr = (int*)malloc(sizeof(int));


**Calloc**

Purpose: Allocates a block of memory for a specified number of elements, each of a specified size, and initializes all bytes to zero. Heap

Initialization: The allocated memory is initialized to zero.

Syntax: void* calloc(size_t num, size_t size);

int *ptr = (int*)calloc(5, sizeof(int));


**Alloc**

Purpose: Allocates memory on the stack, not the heap. In stack frame

Lifetime: The memory is automatically deallocated when the function returns.

Syntax: void* alloca(size_t size);

int *ptr = (int*)alloac(sizeof(int) * 5);


In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0


In [2]:
!nvidia-smi

Wed Jan 29 22:24:40 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Hello CUDA

In [15]:
%%writefile hello_cuda.cu
#include <stdio.h>

__global__ void hello_kernel() {
    printf("Hello from block %d, thread %d!\n", blockIdx.x, threadIdx.x);
}

// blockIdx.x (keyword): returns block Id
// threadIdx.x (keyword): returns thread Id
// blockdim.x (keyword): returns the dimension of the block, number of threads in one block

int main() {
    // Launch kernel with 2 blocks and 4 threads per block
    hello_kernel<<<2, 4>>>();
    cudaDeviceSynchronize();
    return 0;
}


Overwriting hello_cuda.cu


In [16]:
!nvcc -arch=compute_70 -code=sm_70 hello_cuda.cu -o hello_program

In [17]:
! ./hello_program

Hello from block 0, thread 0!
Hello from block 0, thread 1!
Hello from block 0, thread 2!
Hello from block 0, thread 3!
Hello from block 1, thread 0!
Hello from block 1, thread 1!
Hello from block 1, thread 2!
Hello from block 1, thread 3!


## Vector Addition

In [47]:
%%writefile add_vector_cuda.cu
#include<stdio.h>
#include<cuda.h>

// Function/kernel that executes on the GPU
__global__ void arradd(int *x, int *y, int *z)    //kernel definition
{
  int id = blockIdx.x * blockDim.x + threadIdx.x;
/* blockIdx.x gives the respective block id which starts from 0 */
  printf("block %d, thread %d!\n", blockIdx.x, threadIdx.x);
  z[id]=x[id]+y[id];
}

int main()
{
    int a[5];
    int b[5];
    int c[5];
    int *d,*e,*f;


    printf("\n Enter six elements of first array\n");
    for(int i=0;i<5;i++)
    {
        scanf("%d",&a[i]);
    }

    printf("\n Enter six elements of second array\n");
    for(int i=0;i<5;i++)
    {
        scanf("%d",&b[i]);
    }

    /* cudaMalloc() allocates memory from Global memory on GPU */
    cudaMalloc((void **)&d,5*sizeof(int));
    cudaMalloc((void **)&e,5*sizeof(int));
    cudaMalloc((void **)&f,5*sizeof(int));


    /* cudaMemcpy() copies the contents from destination to source. Here destination is GPU(d,e) and source is CPU(a,b) */
    cudaMemcpy(d,a,5*sizeof(int),cudaMemcpyHostToDevice);
    cudaMemcpy(e,b,5*sizeof(int),cudaMemcpyHostToDevice);

    /* call to kernel. Here 2 is number of blocks, 3 is the number of threads per block and d,e,f are the arguments */
    arradd<<<2,3>>>(d,e,f);

    /* Here we are copying content from GPU(Device) to CPU(Host) */
    cudaMemcpy(c,f,5*sizeof(int),cudaMemcpyDeviceToHost);

    printf("\nSum of two arrays:\n ");
    for(int i=0;i<5;i++)
    {
        printf("%d\t",c[i]);
    }

    /* Free the memory allocated to pointers d,e,f */
    cudaFree(d);
    cudaFree(e);
    cudaFree(f);

    return 0;
}

Overwriting add_vector_cuda.cu


In [48]:
!nvcc -arch=compute_70 -code=sm_70 add_vector_cuda.cu -o add

In [49]:
! ./add


 Enter six elements of first array
1
1
1
1
1

 Enter six elements of second array
2
2
2
2
2
block 1, thread 0!
block 1, thread 1!
block 1, thread 2!
block 0, thread 0!
block 0, thread 1!
block 0, thread 2!

Sum of two arrays:
 3	3	3	3	3	